# Data quality and missingness

Inspect class imbalance, missing-value structure, feature ranges and whether missingness itself is associated with APS failures. No model is fit in this notebook.

In [ ]:
from pathlib import Path
import pandas as pd

from scania_aps.data import TRAIN_FILENAME, TEST_FILENAME, read_raw_csv

ROOT = Path.cwd().resolve()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
TRAIN = ROOT / "data" / "raw" / TRAIN_FILENAME
TEST = ROOT / "data" / "raw" / TEST_FILENAME
ARTIFACTS = ROOT / "artifacts"
assert TRAIN.exists() and TEST.exists(), "Run: poetry run scania-aps download"
train = read_raw_csv(TRAIN)
test = read_raw_csv(TEST)
print(train.X.shape, test.X.shape, train.y.mean(), test.y.mean())

In [ ]:
missing = train.X.isna().mean().sort_values(ascending=False)
summary = pd.DataFrame({
    "missing_fraction": missing,
    "median": train.X.median(),
    "mean": train.X.mean(),
})
summary.head(25)

In [ ]:
missing_by_class = train.X.isna().groupby(train.y).mean().T
missing_by_class.columns = ["negative", "positive"]
missing_by_class["difference"] = missing_by_class["positive"] - missing_by_class["negative"]
missing_by_class.reindex(missing_by_class["difference"].abs().sort_values(ascending=False).index).head(25)